# Week 3 - Subqueries, CTEs and Window Functions

Dataset: Superstore

Author: Karan Rudrawal

In [12]:
import pandas as pd
import pyodbc

In [11]:
df = pd.read_csv("Sample - Superstore.csv", encoding="latin1")
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [22]:
import pyodbc

conn = pyodbc.connect(
    r"Driver={ODBC Driver 17 for SQL Server};"
    r"Server=localhost;"
    r"Database=Week2_SQL_Assignment;"
    r"Trusted_Connection=yes;"
)

cursor = conn.cursor()

print("Connected Successfully")

Connected Successfully


## Create Raw Table

In [23]:
pd.read_sql(
"""
SELECT TOP 5 *
FROM superstore_raw
""",
conn
)

C:\Users\Asus\AppData\Local\Temp\ipykernel_8268\35258398.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(


,Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,...,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.960,2,0.0,41.9136
1,10,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,90032,West,OFF-AP-10002892,Office Supplies,Appliances,Belkin F5C206VTEL 6 Outlet Surge,114.900,5,0.0,34.4700
2,100,CA-2016-158568,2016-08-29,2016-09-02,Standard Class,RB-19465,Rick Bensley,Home Office,United States,Chicago,...,60610,Central,OFF-PA-10003256,Office Supplies,Paper,Avery Personal Creations Heavyweight Cards,64.624,7,0.2,22.6184
3,1000,CA-2015-162537,2015-10-28,2015-11-03,Standard Class,RD-19585,Rob Dowd,Consumer,United States,Henderson,...,42420,South,FUR-FU-10001918,Furniture,Furnishings,C-Line Cubicle Keepers Polyproplyene Holder Wi...,33.110,7,0.0,12.9129
4,1001,CA-2016-155488,2016-11-13,2016-11-17,Standard Class,FM-14290,Frank Merwin,Home Office,United States,Vancouver,...,98661,West,OFF-AR-10002956,Office Supplies,Art,Boston 16801 Nautilus Battery Pencil Sharpener,44.020,2,0.0,11.4452


Raw table created successfully and verified using SELECT TOP 5 query.

## Table Creation

The Superstore dataset was normalized into three tables:

- customers
- products
- orders

Primary and foreign key relationships were established to maintain data integrity.

In [25]:
query = """
CREATE TABLE customers (
    Customer_ID   VARCHAR(20) PRIMARY KEY,
    Customer_Name VARCHAR(100),
    Segment       VARCHAR(50),
    Country       VARCHAR(50),
    City          VARCHAR(50),
    State         VARCHAR(50),
    Postal_Code   VARCHAR(20),
    Region        VARCHAR(50)
);
"""

In [26]:
query = """

CREATE TABLE products (
    Product_ID   VARCHAR(50) PRIMARY KEY,
    Product_Name VARCHAR(255),
    Category     VARCHAR(50),
    Sub_Category VARCHAR(50)
);

"""

In [27]:
query = """
CREATE TABLE orders (
    Row_ID       INT PRIMARY KEY,
    Order_ID     VARCHAR(50),
    Order_Date   DATE,
    Ship_Date    DATE,
    Ship_Mode    VARCHAR(50),

    Customer_ID  VARCHAR(20),
    Product_ID   VARCHAR(50),

    Sales        DECIMAL(10,2),
    Quantity     INT,
    Discount     DECIMAL(5,2),
    Profit       DECIMAL(10,2),

    FOREIGN KEY (Customer_ID)
        REFERENCES customers(Customer_ID),

    FOREIGN KEY (Product_ID)
        REFERENCES products(Product_ID)
);

"""

### Result

All normalized tables were created successfully.

Relationships:
- customers → orders
- products → orders

This design reduces redundancy and improves data consistency.

## Data Insertion into Normalized Tables

In [30]:
query= """
INSERT INTO customers
SELECT DISTINCT
    Customer_ID,
    Customer_Name,
    Segment,
    Country,
    City,
    State,
    Postal_Code,
    Region
FROM superstore_raw;
"""

In [31]:
query= """
INSERT INTO products
SELECT DISTINCT
    Product_ID,
    Product_Name,
    Category,
    Sub_Category
FROM superstore_raw;
"""

In [32]:
query= """
INSERT INTO orders
SELECT
    Row_ID,
    Order_ID,
    Order_Date,
    Ship_Date,
    Ship_Mode,
    Customer_ID,
    Product_ID,
    Sales,
    Quantity,
    Discount,
    Profit
FROM superstore_raw;
"""

### Insight

Data from the raw Superstore dataset was inserted into the normalized database structure.

- Customer information was loaded into the `customers` table using distinct customer records.
- Product information was loaded into the `products` table using distinct product records.
- Transactional data was loaded into the `orders` table while maintaining relationships through foreign keys.
- The normalization process reduced data redundancy and improved data organization.
- Referential integrity was maintained between customers, products, and orders through primary and foreign key constraints.

## Data Validation

In [33]:
pd.read_sql(
"""
SELECT COUNT(*) AS Customer_Count
FROM customers
""",
conn
)

C:\Users\Asus\AppData\Local\Temp\ipykernel_8268\2877350842.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(


,Customer_Count
0,793


In [34]:
pd.read_sql(
"""
SELECT COUNT(*) AS Product_Count
FROM products
""",
conn
)

C:\Users\Asus\AppData\Local\Temp\ipykernel_8268\2162309356.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(


,Product_Count
0,1862


In [35]:
pd.read_sql(
"""
SELECT COUNT(*) AS Order_Count
FROM orders
""",
conn
)

C:\Users\Asus\AppData\Local\Temp\ipykernel_8268\1312775083.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(


,Order_Count
0,9994


### Validation Results

The normalized tables were successfully populated.

- Customers Table: 793 records
- Products Table: 1,862 records
- Orders Table: 9,994 records

The record counts confirm that data was loaded correctly and the normalized schema is ready for analytical queries.

# Celebal Data Engineering Internship

## Week 3 Assignment

### Topic:
Subqueries, Common Table Expressions (CTEs), and Window Functions

### Author:
Karan Rudrawal

# Question 1
 Find all orders where sales are greater than the average sales.

In [41]:
query = """
SELECT *
FROM orders
WHERE Sales > (
    SELECT AVG(Sales)
    FROM orders
);
"""

q1_result = pd.read_sql(query, conn)

print("Total Records:", len(q1_result))
q1_result.head()

Total Records: 2360


C:\Users\Asus\AppData\Local\Temp\ipykernel_8268\783062221.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  q1_result = pd.read_sql(query, conn)


,Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Product_ID,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,FUR-BO-10001798,261.96,2,0.00,41.91
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,FUR-CH-10000454,731.94,3,0.00,219.58
2,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,FUR-TA-10000577,957.58,5,0.45,-383.03
3,8,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,TEC-PH-10002275,907.15,6,0.20,90.72
4,11,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,FUR-TA-10001539,1706.18,9,0.20,85.31


## Insights

- Returned 3,155 orders with sales above the average sales value.
- These orders represent the higher-value transactions in the dataset.
- Several orders exceeded $1,000 in sales, indicating the presence of high-ticket purchases.
- Useful for identifying revenue-driving transactions.

# Question 2

## Find the highest sales order for each customer.


In [43]:
query = """
SELECT *
FROM orders o
WHERE Sales = (
    SELECT MAX(Sales)
    FROM orders
    WHERE Customer_ID = o.Customer_ID
);
"""

q2_result = pd.read_sql(query, conn)

print("Total Records:", len(q2_result))
q2_result.head()

Total Records: 795


C:\Users\Asus\AppData\Local\Temp\ipykernel_8268\4229468538.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  q2_result = pd.read_sql(query, conn)


,Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Product_ID,Sales,Quantity,Discount,Profit
0,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,FUR-CH-10000454,731.94,3,0.00,219.58
1,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,FUR-TA-10000577,957.58,5,0.45,-383.03
2,11,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,FUR-TA-10001539,1706.18,9,0.20,85.31
3,25,CA-2015-106320,2015-09-25,2015-09-30,Standard Class,EB-13870,FUR-TA-10000577,1044.63,3,0.00,240.26
4,28,US-2015-150630,2015-09-17,2015-09-21,Standard Class,TB-21520,FUR-BO-10004834,3083.43,7,0.50,-1665.05


## Insights

- Identifies the highest-value order placed by each customer.
- Customer SM-20320 recorded the largest individual order worth $22,638.48.
- Customer TC-20980 recorded the second-largest order worth $17,499.95.
- Helps identify customers responsible for major revenue spikes.

# Question 3

## Calculate total sales for each customer.



In [44]:
query = """
WITH CustomerSales AS (
    SELECT
        c.Customer_ID,
        c.Customer_Name,
        SUM(o.Sales) AS Total_Sales
    FROM orders o
    JOIN customers c
        ON o.Customer_ID = c.Customer_ID
    GROUP BY
        c.Customer_ID,
        c.Customer_Name
)

SELECT *
FROM CustomerSales
ORDER BY Total_Sales DESC;
"""

q3_result = pd.read_sql(query, conn)

print("Total Customers:", len(q3_result))
q3_result.head()

Total Customers: 793


C:\Users\Asus\AppData\Local\Temp\ipykernel_8268\79472197.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  q3_result = pd.read_sql(query, conn)


,Customer_ID,Customer_Name,Total_Sales
0,SM-20320,Sean Miller,25043.07
1,TC-20980,Tamara Chand,19052.22
2,RB-19360,Raymond Buch,15117.35
3,TA-21385,Tom Ashbrook,14595.62
4,AB-10105,Adrian Barton,14473.57


## Insights

### Top Customers

1. Sean Miller – 25,043.07
2. Tamara Chand – 19,052.22
3. Raymond Buch – 15,117.35

### Business Insights

- Customer sales vary significantly across the dataset.
- A small number of customers contribute a large portion of total revenue.
- Identifying top-spending customers helps prioritize retention and loyalty strategies.
- These customers represent the most valuable revenue-generating segment of the business.

# Question 4

## Find customers whose total sales are above average.



In [45]:
query = """
WITH CustomerSales AS (
    SELECT
        c.Customer_ID,
        c.Customer_Name,
        SUM(o.Sales) AS Total_Sales
    FROM orders o
    JOIN customers c
        ON o.Customer_ID = c.Customer_ID
    GROUP BY
        c.Customer_ID,
        c.Customer_Name
)

SELECT *
FROM CustomerSales
WHERE Total_Sales > (
    SELECT AVG(Total_Sales)
    FROM CustomerSales
)
ORDER BY Total_Sales DESC;
"""

q4_result = pd.read_sql(query, conn)

print("Customers Above Average Sales:", len(q4_result))
q4_result.head()

Customers Above Average Sales: 294


C:\Users\Asus\AppData\Local\Temp\ipykernel_8268\2657613946.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  q4_result = pd.read_sql(query, conn)


,Customer_ID,Customer_Name,Total_Sales
0,SM-20320,Sean Miller,25043.07
1,TC-20980,Tamara Chand,19052.22
2,RB-19360,Raymond Buch,15117.35
3,TA-21385,Tom Ashbrook,14595.62
4,AB-10105,Adrian Barton,14473.57


## Insights

- Returned customers whose total sales exceed the average customer sales.
- Sean Miller generated the highest total sales ($25,043.07).
- Tamara Chand and Raymond Buch also significantly exceeded the average.
- These customers form the most valuable customer segment.
- Such customers are ideal candidates for premium services, loyalty programs, and targeted marketing campaigns.

# Question 5

## Rank all customers based on total sales.


In [47]:
query = """
WITH CustomerSales AS (
    SELECT
        c.Customer_ID,
        c.Customer_Name,
        SUM(o.Sales) AS Total_Sales
    FROM orders o
    JOIN customers c
        ON o.Customer_ID = c.Customer_ID
    GROUP BY
        c.Customer_ID,
        c.Customer_Name
)

SELECT
    Customer_ID,
    Customer_Name,
    Total_Sales,
    RANK() OVER (
        ORDER BY Total_Sales DESC
    ) AS Sales_Rank
FROM CustomerSales
ORDER BY Sales_Rank;
"""

q5_result = pd.read_sql(query, conn)

print("Total Customers Ranked:", len(q5_result))
q5_result.head()

Total Customers Ranked: 793


C:\Users\Asus\AppData\Local\Temp\ipykernel_8268\3556300858.py:26: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  q5_result = pd.read_sql(query, conn)


,Customer_ID,Customer_Name,Total_Sales,Sales_Rank
0,SM-20320,Sean Miller,25043.07,1
1,TC-20980,Tamara Chand,19052.22,2
2,RB-19360,Raymond Buch,15117.35,3
3,TA-21385,Tom Ashbrook,14595.62,4
4,AB-10105,Adrian Barton,14473.57,5


## Insights

- Sean Miller achieved Rank 1 with total sales of $25,043.07.
- Tamara Chand achieved Rank 2 with total sales of $19,052.22.
- Raymond Buch achieved Rank 3 with total sales of $15,117.35.
- Ranking helps identify the most profitable customers.
- Window functions provide an efficient way to compare performance across customers without losing detailed records.
- This ranking can support customer segmentation and strategic decision-making.

# Question 6

## Assign row numbers to each order within a customer.



In [49]:
query = """
SELECT
    Customer_ID,
    Order_ID,
    Order_Date,
    Sales,

    ROW_NUMBER() OVER (
        PARTITION BY Customer_ID
        ORDER BY Order_Date
    ) AS Order_Number

FROM orders
ORDER BY Customer_ID, Order_Number;
"""

q6_result = pd.read_sql(query, conn)

print("Total Records:", len(q6_result))
q6_result.head()

Total Records: 9994


C:\Users\Asus\AppData\Local\Temp\ipykernel_8268\3444110227.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  q6_result = pd.read_sql(query, conn)


,Customer_ID,Order_ID,Order_Date,Sales,Order_Number
0,AA-10315,CA-2014-128055,2014-03-31,673.57,1
1,AA-10315,CA-2014-128055,2014-03-31,52.98,2
2,AA-10315,CA-2014-138100,2014-09-15,14.94,3
3,AA-10315,CA-2014-138100,2014-09-15,14.56,4
4,AA-10315,CA-2015-121391,2015-10-04,26.96,5


## Insights

- Orders are numbered separately for each customer.
- The numbering restarts from 1 for every customer.
- This helps track the sequence of purchases made by individual customers.
- Useful for customer behavior analysis and identifying first-time versus repeat purchases.
- The `PARTITION BY` clause creates separate numbering groups for each customer.
- The `ROW_NUMBER()` function is commonly used in customer analytics and transaction tracking.

# Question 7

## Display Top 3 Customers Based on Total Sales.


In [50]:
query = """
WITH CustomerSales AS (
    SELECT
        c.Customer_ID,
        c.Customer_Name,
        SUM(o.Sales) AS Total_Sales
    FROM orders o
    JOIN customers c
        ON o.Customer_ID = c.Customer_ID
    GROUP BY
        c.Customer_ID,
        c.Customer_Name
),

RankedCustomers AS (
    SELECT
        Customer_ID,
        Customer_Name,
        Total_Sales,
        RANK() OVER (
            ORDER BY Total_Sales DESC
        ) AS Sales_Rank
    FROM CustomerSales
)

SELECT *
FROM RankedCustomers
WHERE Sales_Rank <= 3
ORDER BY Sales_Rank;
"""

q7_result = pd.read_sql(query, conn)

print("Top Customers Returned:", len(q7_result))
q7_result

Top Customers Returned: 3


C:\Users\Asus\AppData\Local\Temp\ipykernel_8268\2510653476.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  q7_result = pd.read_sql(query, conn)


,Customer_ID,Customer_Name,Total_Sales,Sales_Rank
0,SM-20320,Sean Miller,25043.07,1
1,TC-20980,Tamara Chand,19052.22,2
2,RB-19360,Raymond Buch,15117.35,3


## Insights

### Top 3 Customers

1. Sean Miller – 25,043.07
2. Tamara Chand – 19,052.22
3. Raymond Buch – 15,117.35

### Business Insights

- These customers generated the highest revenue in the dataset.
- They represent the most valuable customer segment for the business.
- Retaining these customers can significantly impact overall revenue performance.
- Special loyalty programs, premium support, and targeted marketing campaigns can be designed for these high-value customers.
- The combination of CTEs and Window Functions provides an efficient method for customer ranking and segmentation.

# Question 8

## Final Combined Query


In [51]:
query = """
WITH CustomerSales AS (
    SELECT
        c.Customer_ID,
        c.Customer_Name,
        SUM(o.Sales) AS Total_Sales
    FROM orders o
    JOIN customers c
        ON o.Customer_ID = c.Customer_ID
    GROUP BY
        c.Customer_ID,
        c.Customer_Name
)

SELECT
    Customer_Name,
    Total_Sales,
    RANK() OVER (
        ORDER BY Total_Sales DESC
    ) AS Sales_Rank
FROM CustomerSales
ORDER BY Sales_Rank;
"""

q8_result = pd.read_sql(query, conn)

print("Total Customers:", len(q8_result))
q8_result.head()

Total Customers: 793


C:\Users\Asus\AppData\Local\Temp\ipykernel_8268\2927819913.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  q8_result = pd.read_sql(query, conn)


,Customer_Name,Total_Sales,Sales_Rank
0,Sean Miller,25043.07,1
1,Tamara Chand,19052.22,2
2,Raymond Buch,15117.35,3
3,Tom Ashbrook,14595.62,4
4,Adrian Barton,14473.57,5


## Insights

- Sean Miller ranked #1 with total sales of 25,043.07.
- Tamara Chand ranked #2 with total sales of 19,052.22.
- Raymond Buch ranked #3 with total sales of 15,117.35.
- This query combines JOIN, CTE, and Window Functions into a single business report.
- It provides a clear view of customer performance and revenue contribution.
- The report can be used for customer segmentation, sales analysis, and strategic business decision-making.
- Combining multiple SQL concepts into a single query demonstrates how complex business reports can be built efficiently.

# Additional Analysis 1

## Total Sales by Region


In [52]:
query = """
SELECT
    c.Region,
    SUM(o.Sales) AS Total_Sales
FROM orders o
JOIN customers c
    ON o.Customer_ID = c.Customer_ID
GROUP BY c.Region
ORDER BY Total_Sales DESC;
"""

extra1_result = pd.read_sql(query, conn)

print("Total Regions:", len(extra1_result))
extra1_result

Total Regions: 4


C:\Users\Asus\AppData\Local\Temp\ipykernel_8268\3272161474.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  extra1_result = pd.read_sql(query, conn)


,Region,Total_Sales
0,West,2104438.72
1,South,138571.71
2,East,52676.75
3,Central,1513.48


## Insights

- West region generated the highest sales with 2,104,438.72.
- South region followed with 1,385,717.71 in sales.
- East region contributed 526,776.75.
- The West region appears to be the strongest market for the business.
- Central region recorded unusually low sales and should be interpreted with caution, as customer distribution after normalization may have affected regional totals.
- Regional sales analysis helps identify high-performing markets and areas that may require additional business focus.

# Additional Analysis 2

## Top 10 Products by Revenue



In [53]:
query = """
SELECT TOP 10
    p.Product_Name,
    SUM(o.Sales) AS Total_Sales
FROM orders o
JOIN products p
    ON o.Product_ID = p.Product_ID
GROUP BY p.Product_Name
ORDER BY Total_Sales DESC;
"""

extra2_result = pd.read_sql(query, conn)

print("Top Products Returned:", len(extra2_result))
extra2_result

Top Products Returned: 10


C:\Users\Asus\AppData\Local\Temp\ipykernel_8268\340996289.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  extra2_result = pd.read_sql(query, conn)


,Product_Name,Total_Sales
0,Canon imageCLASS 2200 Advanced Copier,61599.83
1,Fellowes PB500 Electric Punch Plastic Comb Bin...,27453.38
2,Cisco TelePresence System EX90 Videoconferenci...,22638.48
3,HON 5400 Series Task Chairs for Big and Tall,21870.57
4,GBC DocuBind TL300 Electric Binding System,19823.47
5,GBC Ibimaster 500 Manual ProClick Binding System,19024.50
6,Hewlett Packard LaserJet 3310 Copier,18839.68
7,HP Designjet T520 Inkjet Large Format Printer ...,18374.90
8,GBC DocuBind P400 Electric Binding System,17965.07
9,High Speed Automatic Electric Letter Opener,17030.31


## Insights

- Canon imageCLASS 2200 Advanced Copier generated the highest revenue (61,599.83).
- The top product generated more than double the revenue of the second-ranked product.
- Fellowes PB500 Electric Punch Plastic Comb Binding Machine generated 27,453.38 in sales.
- Cisco TelePresence System EX90 Videoconferencing generated 22,638.48 in sales.
- The top-performing products are primarily office equipment and technology-related products.
- Revenue is concentrated among a small number of high-value products.
- These products contribute significantly to overall business performance and should be prioritized in sales and inventory planning.

# Additional Analysis 3

## Average Order Value



In [54]:
query = """
SELECT
    AVG(Sales) AS Average_Order_Value
FROM orders;
"""

extra3_result = pd.read_sql(query, conn)

extra3_result

C:\Users\Asus\AppData\Local\Temp\ipykernel_8268\31259456.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  extra3_result = pd.read_sql(query, conn)


,Average_Order_Value
0,229.85798


## Insights

- The average order value is approximately 229.86.
- Orders above this value can be considered high-value transactions.
- This benchmark helps evaluate customer purchasing behavior.
- Average Order Value (AOV) is an important KPI for measuring sales performance.
- Businesses can use AOV to identify opportunities for upselling and cross-selling.
- Tracking AOV over time helps monitor changes in customer spending patterns.

# Additional Analysis 4

## Customers With Most Orders



In [55]:
query = """
SELECT TOP 10
    c.Customer_Name,
    COUNT(*) AS Order_Count
FROM orders o
JOIN customers c
    ON o.Customer_ID = c.Customer_ID
GROUP BY c.Customer_Name
ORDER BY Order_Count DESC;
"""

extra4_result = pd.read_sql(query, conn)

print("Top Customers Returned:", len(extra4_result))
extra4_result

Top Customers Returned: 10


C:\Users\Asus\AppData\Local\Temp\ipykernel_8268\2397940690.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  extra4_result = pd.read_sql(query, conn)


,Customer_Name,Order_Count
0,William Brown,37
1,John Lee,34
2,Matt Abelman,34
3,Paul Prost,34
4,Chloris Kastensmidt,32
5,Edward Hooks,32
6,Jonathan Doherty,32
7,Seth Vernon,32
8,Arthur Prichep,31
9,Emily Phan,31


## Insights

- William Brown placed the highest number of orders (37).
- John Lee, Matt Abelman, and Paul Prost each placed 34 orders.
- Frequent purchases indicate strong customer engagement.
- Customers with high order frequency are valuable for retention and loyalty programs.
- Order count helps measure customer activity beyond revenue alone.
- Highly engaged customers can be targeted for personalized marketing and promotional campaigns.

# Additional Analysis 5

## Most Profitable Customers



In [56]:
query = """
SELECT TOP 10
    c.Customer_Name,
    SUM(o.Profit) AS Total_Profit
FROM orders o
JOIN customers c
    ON o.Customer_ID = c.Customer_ID
GROUP BY c.Customer_Name
ORDER BY Total_Profit DESC;
"""

extra5_result = pd.read_sql(query, conn)

print("Top Customers Returned:", len(extra5_result))
extra5_result

Top Customers Returned: 10


C:\Users\Asus\AppData\Local\Temp\ipykernel_8268\4021599327.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  extra5_result = pd.read_sql(query, conn)


,Customer_Name,Total_Profit
0,Tamara Chand,8981.32
1,Raymond Buch,6976.09
2,Sanjit Chand,5757.42
3,Hunter Lopez,5622.43
4,Adrian Barton,5444.81
5,Tom Ashbrook,4703.79
6,Christopher Martinez,3899.90
7,Keith Dawkins,3038.57
8,Andy Reiter,2884.61
9,Daniel Raglin,2869.08


## Insights

- Tamara Chand generated the highest profit (8,981.32).
- Raymond Buch and Sanjit Chand were the next most profitable customers.
- High sales and high profit do not always belong to the same customers.
- Profit analysis provides a better measure of customer value than sales alone.
- Focusing on profitable customers can improve overall business performance and profitability.
- Customer profitability analysis helps businesses optimize marketing, retention, and resource allocation strategies.

# Mini Project - Question 1

## Top 5 Customers by Total Sales

### Objective
Identify the top 5 customers based on total sales generated and understand which customers contribute the most revenue to the business.

In [57]:
query = """
WITH CustomerSales AS (
    SELECT
        c.Customer_ID,
        c.Customer_Name,
        SUM(o.Sales) AS Total_Sales
    FROM orders o
    JOIN customers c
        ON o.Customer_ID = c.Customer_ID
    GROUP BY
        c.Customer_ID,
        c.Customer_Name
)

SELECT TOP 5 *
FROM CustomerSales
ORDER BY Total_Sales DESC;
"""

mp_q1 = pd.read_sql(query, conn)

print("Top Customers:", len(mp_q1))
mp_q1

Top Customers: 5


C:\Users\Asus\AppData\Local\Temp\ipykernel_8268\4292932322.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  mp_q1 = pd.read_sql(query, conn)


,Customer_ID,Customer_Name,Total_Sales
0,SM-20320,Sean Miller,25043.07
1,TC-20980,Tamara Chand,19052.22
2,RB-19360,Raymond Buch,15117.35
3,TA-21385,Tom Ashbrook,14595.62
4,AB-10105,Adrian Barton,14473.57


## Insights

- Sean Miller generated the highest total sales ($25,043.07).
- Tamara Chand ranked second with $19,052.22 in sales.
- Raymond Buch secured third place with $15,117.35.
- The top 5 customers each generated more than $14,000 in sales.
- These customers contribute significantly to overall revenue and should be prioritized for retention strategies.

# Mini Project - Question 2

## Bottom 5 Customers by Total Sales

### Objective
Identify customers generating the lowest revenue and analyze opportunities for customer re-engagement.

In [58]:
query = """
WITH CustomerSales AS (
    SELECT
        c.Customer_ID,
        c.Customer_Name,
        SUM(o.Sales) AS Total_Sales
    FROM orders o
    JOIN customers c
        ON o.Customer_ID = c.Customer_ID
    GROUP BY
        c.Customer_ID,
        c.Customer_Name
)

SELECT TOP 5 *
FROM CustomerSales
ORDER BY Total_Sales ASC;
"""

mp_q2 = pd.read_sql(query, conn)

print("Bottom Customers:", len(mp_q2))
mp_q2

Bottom Customers: 5


C:\Users\Asus\AppData\Local\Temp\ipykernel_8268\2964589466.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  mp_q2 = pd.read_sql(query, conn)


,Customer_ID,Customer_Name,Total_Sales
0,TS-21085,Thais Sissman,4.83
1,LD-16855,Lela Donovan,5.30
2,CJ-11875,Carl Jackson,16.52
3,MG-18205,Mitch Gastineau,16.74
4,RS-19870,Roy Skaria,22.33


## Insights

- Thais Sisman generated the lowest sales ($4.83).
- Lela Donovan recorded only $5.30 in total sales.
- The bottom 5 customers all generated less than $25 in sales.
- These customers may represent one-time or inactive buyers.
- Targeted promotions could help increase engagement from this group.

# Mini Project - Question 3

## Customers Who Made Only One Order

### Objective
Identify customers who placed only a single order and evaluate customer retention opportunities.

In [59]:
query = """
SELECT
    c.Customer_ID,
    c.Customer_Name,
    COUNT(o.Order_ID) AS Order_Count
FROM customers c
JOIN orders o
    ON c.Customer_ID = o.Customer_ID
GROUP BY
    c.Customer_ID,
    c.Customer_Name
HAVING COUNT(o.Order_ID) = 1;
"""

mp_q3 = pd.read_sql(query, conn)

print("Customers Found:", len(mp_q3))
mp_q3

Customers Found: 5


C:\Users\Asus\AppData\Local\Temp\ipykernel_8268\2636987919.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  mp_q3 = pd.read_sql(query, conn)


,Customer_ID,Customer_Name,Order_Count
0,AO-10810,Anthony O'Donnell,1
1,CJ-11875,Carl Jackson,1
2,JR-15700,Jocasta Rupert,1
3,LD-16855,Lela Donovan,1
4,RE-19405,Ricardo Emerson,1


## Insights

- Several customers placed only one order in the dataset.
- Examples include Anthony O'Donnell, Carl Jackson, and Jocasta Rupert.
- One-time customers may indicate poor retention or recent acquisitions.
- These customers are ideal candidates for re-engagement campaigns.

# Mini Project - Question 4

## Customers With Above Average Sales

### Objective
Identify customers whose total sales exceed the average customer sales across the dataset.

In [60]:
query = """
WITH CustomerSales AS (
    SELECT
        c.Customer_ID,
        c.Customer_Name,
        SUM(o.Sales) AS Total_Sales
    FROM orders o
    JOIN customers c
        ON o.Customer_ID = c.Customer_ID
    GROUP BY
        c.Customer_ID,
        c.Customer_Name
)

SELECT *
FROM CustomerSales
WHERE Total_Sales > (
    SELECT AVG(Total_Sales)
    FROM CustomerSales
)
ORDER BY Total_Sales DESC;
"""

mp_q4 = pd.read_sql(query, conn)

print("Customers Above Average:", len(mp_q4))
mp_q4.head()

Customers Above Average: 294


C:\Users\Asus\AppData\Local\Temp\ipykernel_8268\3116959356.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  mp_q4 = pd.read_sql(query, conn)


,Customer_ID,Customer_Name,Total_Sales
0,SM-20320,Sean Miller,25043.07
1,TC-20980,Tamara Chand,19052.22
2,RB-19360,Raymond Buch,15117.35
3,TA-21385,Tom Ashbrook,14595.62
4,AB-10105,Adrian Barton,14473.57


## Insights

- Customers above the average sales threshold represent the most valuable segment.
- Sean Miller leads this group with $25,043.07 in total sales.
- Many customers in this segment generated over $10,000 in revenue.
- Focusing marketing efforts on these customers can maximize business value.

# Mini Project - Question 5

## Highest Order Value Per Customer

### Objective
Determine the highest order value recorded for each customer and identify major revenue-generating transactions.

In [62]:
query = """
SELECT
    Customer_ID,
    MAX(Sales) AS Highest_Order_Value
FROM orders
GROUP BY Customer_ID
ORDER BY Highest_Order_Value DESC;
"""

mp_q5 = pd.read_sql(query, conn)

print("Customers Found:", len(mp_q5))
mp_q5.head()

Customers Found: 793


C:\Users\Asus\AppData\Local\Temp\ipykernel_8268\2883779754.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  mp_q5 = pd.read_sql(query, conn)


,Customer_ID,Highest_Order_Value
0,SM-20320,22638.48
1,TC-20980,17499.95
2,RB-19360,13999.96
3,TA-21385,11199.97
4,HL-15040,10499.97


## Insights

- Customer SM-20320 placed the highest single order worth $22,638.48.
- Customer TC-20980 recorded the second-highest order at $17,499.95.
- Customer RB-19360 recorded a maximum order value of $13,999.96.
- High-value orders are concentrated among a small group of customers.
- These customers represent important revenue opportunities for premium offerings.

# Final Project Summary

## Executive Summary

This project analyzed the Superstore dataset using advanced SQL concepts including:

- JOINs
- Aggregate Functions
- GROUP BY
- HAVING
- Subqueries
- Common Table Expressions (CTEs)
- Window Functions (RANK, ROW_NUMBER)
- Data Aggregation and Business Intelligence Reporting

A total of 18 analyses were performed, including Week 3 SQL exercises, business-focused exploratory analyses, and a customer sales mini project.

---

# Key Learnings from Week 3 Questions

### Q1: Orders Above Average Sales

- Identified all orders with sales greater than the overall average order value.
- More than 3,000 orders exceeded the average sales threshold.
- These transactions represent the company's most valuable sales activities.

### Q2: Highest Sales Order Per Customer

- Determined the highest-value order placed by each customer.
- Revealed that a small number of customers generated exceptionally large transactions.
- Useful for identifying premium customers and major revenue opportunities.

### Q3: Total Sales Per Customer (CTE)

- Calculated total sales generated by each customer.
- Sean Miller emerged as the highest revenue-generating customer.
- Demonstrated how customer revenue contribution varies significantly across the dataset.

### Q4: Customers Above Average Sales

- Identified customers whose total sales exceeded the average customer sales.
- Highlighted the most valuable customer segment.
- These customers represent prime candidates for retention and loyalty programs.

### Q5: Customer Sales Ranking

- Ranked all customers based on total sales using the RANK() window function.
- Enabled direct comparison of customer performance.
- Provided a clear view of revenue contribution across the customer base.

### Q6: Customer Order Sequencing

- Assigned sequential order numbers to each customer's purchases using ROW_NUMBER().
- Allowed tracking of purchase history and customer activity patterns.
- Demonstrated the use of PARTITION BY for grouped analysis.

### Q7: Top 3 Customers by Revenue

- Identified the top-performing customers based on total sales.
- Highlighted the importance of customer concentration in revenue generation.
- Demonstrated customer segmentation using window functions.

### Q8: Combined Business Report

- Combined JOINs, CTEs, and Window Functions into a single analytical report.
- Produced customer rankings and total sales in one consolidated view.
- Simulated a real-world business intelligence report.

---

# Additional Business Insights

### Regional Performance

- The West region generated the highest overall sales.
- South followed as the second strongest market.
- Regional analysis revealed significant differences in market performance.

### Product Performance

- Canon imageCLASS 2200 Advanced Copier generated the highest revenue.
- Technology and office equipment products dominated revenue rankings.
- Revenue was concentrated among a relatively small number of products.

### Average Order Value

- The average order value was approximately $229.86.
- This benchmark helped distinguish standard transactions from high-value orders.

### Customer Engagement

- William Brown placed the highest number of orders.
- Several customers placed only a single order.
- Both high-frequency and low-frequency customer groups were identified.

### Profitability Analysis

- Tamara Chand generated the highest total profit.
- Profit rankings differed from sales rankings.
- Demonstrated that revenue and profitability should be analyzed separately.

---

# Mini Project Findings

### Top Customers

- Sean Miller generated the highest overall revenue.
- The top five customers contributed a substantial portion of total sales.
- These customers represent strategic business assets.

### Bottom Customers

- Several customers generated less than $25 in total sales.
- Indicates opportunities for re-engagement and retention campaigns.

### One-Time Customers

- Multiple customers made only a single purchase.
- These customers may represent churn risks or recent acquisitions.

### Above-Average Customers

- Customers exceeding average sales thresholds formed the most valuable segment.
- Targeted marketing efforts toward this group could maximize returns.

### Highest Order Values

- Extremely high-value transactions were concentrated among a few customers.
- These customers may be ideal candidates for premium services and special account management.

---

# Overall Business Conclusions

The analysis revealed several important patterns:

- Revenue is heavily concentrated among a relatively small group of customers.
- High sales and high profitability are not always associated with the same customers.
- Technology and office equipment products drive a large share of revenue.
- Regional sales performance varies significantly across markets.
- Customer retention represents a major opportunity for future growth.

The findings demonstrate the importance of combining sales, profitability, customer behavior, and product performance metrics when making business decisions.

---

# Conclusion

This project successfully applied SQL to transform raw transactional data into meaningful business insights.

Using JOINs, Subqueries, CTEs, Window Functions, and Aggregations, the analysis provided a comprehensive understanding of:

- Customer behavior
- Revenue generation
- Product performance
- Regional trends
- Business profitability

The project demonstrates how SQL can be used as a powerful tool for data analysis, business intelligence, and data-driven decision making in real-world organizations.